# Week 7 — ResNet-18 Fine-Tuning (STARTER)
### The Computer Vision Internship | VisionAI Lagos

## Part 1 — ResNet-18 Architecture

> 🧠 **What Will This Output?**
> Stage 1 trains only the FC head (2,052 params). Stage 2 unfreezes all 11.7M. Predict val F1 after Stage 1 vs Stage 2. Which will be higher — and by how much?

In [ ]:
def build_resnet18(num_classes=4,pretrained=True):
    weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    m=models.resnet18(weights=weights)
    m.fc=nn.Sequential(nn.Dropout(0.4),nn.Linear(m.fc.in_features,num_classes))
    return m

model=build_resnet18().to(DEVICE)
total=sum(p.numel() for p in model.parameters())
head =sum(p.numel() for p in model.fc.parameters())
print(f"Total: {total:,} | Head: {head:,} ({head/total:.1%})")

In [ ]:
# Stage 1: freeze backbone, train head only (5 epochs)
for name,param in model.named_parameters():
    param.requires_grad = "fc" in name
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Stage 1: {trainable:,} trainable (head only)")
history1,best1,ep1=run_training(model,df,n_epochs=5,lr_head=1e-3,save_path=None,verbose=True)
print(f"Stage 1 best val F1: {best1:.3f}")

In [ ]:
# Stage 2: unfreeze all with differential LRs
for param in model.parameters(): param.requires_grad=True
print("Stage 2: all layers unfrozen")
print("backbone lr=1e-5 (treat ImageNet weights gently)")
print("head lr=1e-4")
history2,best2,ep2=run_training(
    model,df,n_epochs=15,lr_head=1e-4,lr_backbone=1e-5,
    save_path="models/resnet18_best.pth",verbose=True
)
print(f"Stage 2 best val F1: {best2:.3f}")
print(f"Improvement over Stage 1: +{best2-best1:.3f}")

## Part 2 — Feature Map Comparison

> ⏸️ **Recall Prompt**
> The ResNet features are more structured than scratch CNN features. In your own words: why does a model pretrained on 14M ImageNet images produce better feature maps for Nigerian satellite patches, even though it has never seen a satellite image?

In [ ]:
# Load ImageNet pretrained ResNet (no fine-tuning)
resnet_pt=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).to(DEVICE).eval()
ev=transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
row=df[(df["city"]=="Kano")&(df["land_use"]=="industrial")].iloc[0]
img_pil=Image.open(f"datasets/images/{row['filename']}").convert("RGB")
img_t=ev(img_pil).unsqueeze(0).to(DEVICE)

fmaps_pt={}
def hook(name): return lambda m,i,o: fmaps_pt.update({name:o.detach().cpu()})
h=resnet_pt.layer1.register_forward_hook(hook("layer1"))
with torch.no_grad(): resnet_pt(img_t)
h.remove()

fig,axes=plt.subplots(1,9,figsize=(20,3))
axes[0].imshow(np.array(img_pil)); axes[0].set_title("Kano Industrial"); axes[0].axis("off")
fmap=fmaps_pt["layer1"]
for ci in range(1,9):
    fm=fmap[0,ci-1].numpy(); fm=(fm-fm.min())/(fm.max()-fm.min()+1e-8)
    axes[ci].imshow(fm,cmap="viridis"); axes[ci].set_title(f"Ch{ci}"); axes[ci].axis("off")
plt.suptitle("ResNet-18 ImageNet Features (Layer 1) — More Structured Than Scratch CNN",fontweight="bold")
plt.tight_layout(); plt.savefig("outputs/resnet_features.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()

In [ ]:
# Per-city evaluation
model.load_state_dict(torch.load("models/resnet18_best.pth",map_location=DEVICE))
city_f1=per_city_f1(model,df,DEVICE)
print("Per-city F1 — ResNet-18:")
for city,f1 in sorted(city_f1.items(),key=lambda x:-x[1]):
    bar="█"*int(f1*30); print(f"  {city:<8}: {f1:.3f}  {bar}")
print(f"Equity gap: {max(city_f1.values())-min(city_f1.values()):.3f}")

## Week 7 Complete

Next: `week8_efficientnet_STARTER.ipynb`

*The Computer Vision Internship · VisionAI Lagos*

## ✅ By completing Week 7, you can now:

- Fine-tune ResNet-18 using feature extraction and full fine-tuning
- Compare feature maps between your scratch CNN and ResNet-18
- Explain the skip connection and why it solves the vanishing gradient problem
- Produce a before/after accuracy table showing transfer learning benefit

📤 **GitHub:** Push week7_resnet_STARTER.ipynb. Commit: "Week 7: ResNet-18 fine-tuned, benefit measured"


---

## 📚 Get the Full Book

This notebook is part of **The Computer Vision Internship** — Book 3 of the InternshipInABook™ Series.

👉 **[Get the book on Selar → [SELAR_LINK_PLACEHOLDER]]**

---
*InternshipInABook™ · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook/cv-internship-in-a-book)*
